# SSTW Wan2.1 Flow Adapter Preflight - 真实 GPU 入口

本 notebook 只运行 `flow_model_adapter_preflight`, 用于验证 Wan2.1 是否能加载、callback 是否能捕获 latent、time grid 和 sampler signature 是否可记录、velocity / latent displacement proxy 是否可获得、L4 显存是否足够。

通过前不得运行 sampling-time constraint 小实验或 full generative video probe。


## 运行约束

1. Runtime 类型选择 GPU, 优先 L4。
2. 如模型需要 Hugging Face 权限, 先在环境变量中提供 `HF_TOKEN`。
3. Notebook 不直接写正式 records; 正式输出由仓库模块 `experiments.flow_model_adapter_preflight.wan21_preflight` 写出。
4. 若 `adapter_preflight_decision` 为 `FAIL`, 应先修 adapter, 不进入 B6 sampling-time constraint。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 如果仓库还没有 clone 到 Colab, 请先执行类似命令:
# !git clone <your_repo_url> /content/SSTW
# %cd /content/SSTW

%cd /content/SSTW


In [ ]:
# 安装或更新真实 Wan2.1 preflight 所需依赖。
!pip -q install --upgrade diffusers transformers accelerate safetensors imageio imageio-ffmpeg


In [ ]:
import os

# 推荐在 Colab Secrets 中配置 HF_TOKEN, 这里仅检查是否存在, 不打印 token 值。
print('HF_TOKEN status:', 'provided' if os.environ.get('HF_TOKEN') else 'not_provided')


In [ ]:
!nvidia-smi


In [ ]:
# 先运行默认快速测试和 harness 审计, 确认 Colab 中仓库状态没有破坏治理约束。
!pytest -q
!python tools/harness/run_all_audits.py


In [ ]:
from paper_workflow.notebook_utils.flow_model_adapter_preflight_workflow import (
    DEFAULT_WAN21_PREFLIGHT_MODEL_ID,
    build_wan21_flow_adapter_preflight_command,
    ensure_drive_layout,
    run_command,
)

DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/SSTW'
MODEL_ID = DEFAULT_WAN21_PREFLIGHT_MODEL_ID
NUM_INFERENCE_STEPS = 4
NUM_FRAMES = 33
HEIGHT = 320
WIDTH = 512

layout = ensure_drive_layout(DRIVE_PROJECT_ROOT)
command = build_wan21_flow_adapter_preflight_command(
    layout,
    model_id=MODEL_ID,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_frames=NUM_FRAMES,
    height=HEIGHT,
    width=WIDTH,
)
print(' '.join(command))


In [ ]:
result = run_command(command)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Wan2.1 Flow adapter preflight failed with return code {result.returncode}')


In [ ]:
import json
from pathlib import Path

decision_path = Path(layout['drive_run_root']) / 'artifacts' / 'wan21_flow_adapter_preflight_decision.json'
decision = json.loads(decision_path.read_text(encoding='utf-8'))
print(json.dumps(decision, ensure_ascii=False, indent=2, sort_keys=True))

assert decision['adapter_preflight_decision'] == 'PASS', 'preflight 未通过, 不得进入 B6 sampling-time constraint'
assert decision['model_load_status'] == 'loaded'
assert decision['callback_latent_capture_status'] == 'captured'
assert decision['time_grid_capture_status'] == 'captured'
assert decision['sampler_signature_status'] == 'captured'
assert decision['velocity_proxy_status'] == 'captured'
